# CIS6005 Computational Intelligence
## Notebook 09 — Hyperparameter Tuning
**Student Health Risk Prediction | Kaggle PS S6E7**

---
### What Are Hyperparameters?
Parameters are values the model learns from data. **Hyperparameters** are configuration values WE choose BEFORE training:

| Model | Key Hyperparameters |
|-------|--------------------|
| Random Forest | n_estimators, max_depth, min_samples_split |
| Gradient Boosting | n_estimators, learning_rate, max_depth |

### Why GridSearchCV?
`GridSearchCV` systematically tries **every combination** of hyperparameter values and uses cross-validation to pick the best. This is exhaustive but reliable.

> **Why tune?** Default hyperparameters are generic. Tuning squeezes the last few percentage points of accuracy specific to YOUR dataset.

In [1]:
# ============================================================
# IMPORTS & SETUP
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.ensemble        import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics         import f1_score, accuracy_score, classification_report

PROJECT_ROOT = Path.cwd().parent
PROC_DATA    = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR   = PROJECT_ROOT / 'models'

X_train = np.load(PROC_DATA / 'X_train.npy')
X_val   = np.load(PROC_DATA / 'X_val.npy')
y_train = np.load(PROC_DATA / 'y_train.npy')
y_val   = np.load(PROC_DATA / 'y_val.npy')

X_full = np.vstack([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

# Downsample for hyperparameter tuning grid search (speed up execution)
np.random.seed(42)
idx = np.random.choice(len(X_full), 20000, replace=False)
X_full_sub = X_full[idx]
y_full_sub = y_full[idx]

label_encoder = joblib.load(MODELS_DIR / 'label_encoder.joblib')
CLASS_NAMES   = list(label_encoder.classes_)

RANDOM_STATE = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('✅ Setup complete. Ready for hyperparameter tuning.')

✅ Setup complete. Ready for hyperparameter tuning.


## 1. Tune Random Forest

**Why Random Forest?** It is typically among the top performers on tabular data and its hyperparameters are well-understood and relatively fast to tune.

In [2]:
# ============================================================
# SECTION 1: Random Forest — GridSearchCV
# ============================================================

rf_param_grid = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [10, 15, None],
    'min_samples_split': [5, 10],
    'min_samples_leaf' : [1, 2]
}

print(f'Grid size: {2 * 3 * 2 * 2} combinations × 5 folds = {2*3*2*2*5} fits')
print('⏳ This will take a few seconds... please wait.\n')

rf_base = RandomForestClassifier(
    class_weight='balanced',
    n_jobs=None,  # Avoid nested parallelism with GridSearchCV(n_jobs=-1)
    random_state=RANDOM_STATE
)

rf_grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=rf_param_grid,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1,
    refit=True
)

rf_grid_search.fit(X_full_sub, y_full_sub)

print('\n' + '=' * 55)
print('  Random Forest — Best Hyperparameters')
print('=' * 55)
for param, value in rf_grid_search.best_params_.items():
    print(f'  {param:<25}: {value}')
print(f'  Best CV F1 Score    : {rf_grid_search.best_score_:.4f}')
print('=' * 55)

Grid size: 24 combinations × 5 folds = 120 fits
⏳ This will take a few seconds... please wait.

Fitting 5 folds for each of 24 candidates, totalling 120 fits



  Random Forest — Best Hyperparameters
  max_depth                : None
  min_samples_leaf         : 1
  min_samples_split        : 5
  n_estimators             : 200
  Best CV F1 Score    : 0.9605


## 2. Tune Gradient Boosting

In [3]:
# ============================================================
# SECTION 2: Gradient Boosting — GridSearchCV
# ============================================================

gb_param_grid = {
    'max_iter'     : [100, 150],
    'learning_rate': [0.05, 0.1],
    'max_depth'    : [3, 5]
}

print(f'Grid size: {2*2*2} combinations × 5 folds = {2*2*2*5} fits')
print('⏳ This will take less than a minute... please wait.\n')

gb_base = HistGradientBoostingClassifier(random_state=RANDOM_STATE)

gb_grid_search = GridSearchCV(
    estimator=gb_base,
    param_grid=gb_param_grid,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1,
    refit=True
)

gb_grid_search.fit(X_full_sub, y_full_sub)

print('\n' + '=' * 55)
print('  Gradient Boosting — Best Hyperparameters')
print('=' * 55)
for param, value in gb_grid_search.best_params_.items():
    print(f'  {param:<25}: {value}')
print(f'  Best CV F1 Score    : {gb_grid_search.best_score_:.4f}')
print('=' * 55)

Grid size: 8 combinations × 5 folds = 40 fits
⏳ This will take less than a minute... please wait.

Fitting 5 folds for each of 8 candidates, totalling 40 fits



  Gradient Boosting — Best Hyperparameters
  learning_rate            : 0.05
  max_depth                : 3
  max_iter                 : 150
  Best CV F1 Score    : 0.9610


## 3. Evaluate Tuned Models

In [4]:
# ============================================================
# SECTION 3: Evaluate Tuned Models on Validation Set
# ============================================================

# Refit the best estimators on the full training data (X_full) to get maximum performance
print('Refitting best tuned models on the full dataset...')
rf_best_params = rf_grid_search.best_params_
gb_best_params = gb_grid_search.best_params_

rf_best = RandomForestClassifier(
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    **rf_best_params
)
rf_best.fit(X_full, y_full)

gb_best = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE,
    **gb_best_params
)
gb_best.fit(X_full, y_full)

tuned_models = {
    'Tuned Random Forest'    : rf_best,
    'Tuned Gradient Boosting': gb_best
}

print('=' * 65)
print('  TUNED MODEL VALIDATION PERFORMANCE')
print('=' * 65)
print(f'  {"Model":<30} {"Accuracy":>10} {"F1 (W)":>10} {"F1 (M)":>10}')
print('-' * 65)

tuned_results = {}
for name, model in tuned_models.items():
    y_pred = model.predict(X_val)
    acc    = accuracy_score(y_val, y_pred)
    f1_w   = f1_score(y_val, y_pred, average='weighted')
    f1_m   = f1_score(y_val, y_pred, average='macro')
    
    tuned_results[name] = {'model': model, 'val_acc': acc, 'val_f1_w': f1_w, 'val_f1_m': f1_m}
    print(f'  {name:<30} {acc:>10.4f} {f1_w:>10.4f} {f1_m:>10.4f}')

print('=' * 65)

for name, res in tuned_results.items():
    print(f'\n--- {name} Classification Report ---')
    print(classification_report(y_val, res['model'].predict(X_val), target_names=CLASS_NAMES))

Refitting best tuned models on the full dataset...


  TUNED MODEL VALIDATION PERFORMANCE
  Model                            Accuracy     F1 (W)     F1 (M)
-----------------------------------------------------------------


  Tuned Random Forest                0.9991     0.9991     0.9972


  Tuned Gradient Boosting            0.9654     0.9638     0.9028

--- Tuned Random Forest Classification Report ---


              precision    recall  f1-score   support

     at-risk       1.00      1.00      1.00    118512
         fit       0.99      1.00      1.00      7961
   unhealthy       0.99      1.00      1.00     11545

    accuracy                           1.00    138018
   macro avg       1.00      1.00      1.00    138018
weighted avg       1.00      1.00      1.00    138018


--- Tuned Gradient Boosting Classification Report ---


              precision    recall  f1-score   support

     at-risk       0.97      1.00      0.98    118512
         fit       0.95      0.80      0.87      7961
   unhealthy       0.98      0.77      0.86     11545

    accuracy                           0.97    138018
   macro avg       0.96      0.85      0.90    138018
weighted avg       0.97      0.97      0.96    138018



## 4. Select and Save the Best Tuned Model

In [5]:
# ============================================================
# SECTION 4: Save Best Tuned Model
# ============================================================

# Choose the tuned model with the highest validation F1
best_tuned_name  = max(tuned_results, key=lambda k: tuned_results[k]['val_f1_w'])
best_tuned_model = tuned_results[best_tuned_name]['model']
best_tuned_f1    = tuned_results[best_tuned_name]['val_f1_w']

joblib.dump(best_tuned_model, MODELS_DIR / 'model_tuned_best.joblib')
joblib.dump(rf_grid_search,   MODELS_DIR / 'gridsearch_rf.joblib')
joblib.dump(gb_grid_search,   MODELS_DIR / 'gridsearch_gb.joblib')

print('=' * 60)
print('  PHASE 9 COMPLETE — Hyperparameter Tuning')
print('=' * 60)
print(f'  🏆 Best Tuned Model : {best_tuned_name}')
print(f'  🎯 Validation F1    : {best_tuned_f1:.4f}')
print('=' * 60)
print('  ✅ Saved: models/model_tuned_best.joblib')
print('  ✅ Ready for Phase 10: Model Comparison')
print('=' * 60)

  PHASE 9 COMPLETE — Hyperparameter Tuning
  🏆 Best Tuned Model : Tuned Random Forest
  🎯 Validation F1    : 0.9991
  ✅ Saved: models/model_tuned_best.joblib
  ✅ Ready for Phase 10: Model Comparison
